In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import scipy as sp
from scipy.integrate import odeint
import h5py

In [ ]:
# sp.integrate.odeint

In [ ]:
Nt=8

In [ ]:
Ns=32

# v16 clean-room (32c). UNCOMMENT ONE mass block, run the notebook, then switch to the next mass.
# ibeta_m/ibeta_M = argmin window bracketing the mean Tc; iblo/ibhi = Veff.h5 read range (must cover
# the bracket; get_potential_jk wrote the FULL 0-1000 grid). nbeta = # jk source ensembles (jdrop count).
# v16 mean Tc (argmin (VA-VB)^2 on the mean Veff): m0.4=841, m0.3=500, m0.2=785. jk ibetac clusters
# tightly at the mean (eps=0.5 shrink); verified no bracket edge-hit.

mass="0p4000"
ibeta_m = 816
ibeta_M = 866
nbeta = 9
iblo = 790
ibhi = 890

mass="0p3000"
ibeta_m = 475
ibeta_M = 525
nbeta = 8
iblo = 450
ibhi = 550

mass="0p2000"
ibeta_m = 760
ibeta_M = 810
nbeta = 9
iblo = 735
ibhi = 835

In [ ]:
ibetas=np.arange(iblo, ibhi+1)  # clean-room Veff.h5 window (per-mass)

In [ ]:
directory2 = "/mnt/hdd_barracuda/llnl/reweight/data/"+str(Ns)+"b_v16/"

# v16 realaxis: central row iby = nptsy/2-1 = 39 (nptsy=80); iby0 does not exist. betas grid is the
# same in every meas.bin, so any central-row nojk file works.
f = h5py.File(directory2+"/m"+mass+"avghist_ibx"+str(0)+"_iby"+str(39)+"_nojkmeas.bin", 'r')
betas = f['beta'][()]
f.close()

In [ ]:
ibetac_jk_ = []
for jdrop in np.arange(nbeta):
    tmp_ = []
    for ibin in np.arange(40):
        print(jdrop, ibin)
    # jdrop = 4
    # ibin = 3
        desc = "jk_"+str(jdrop)+"_40_"+str(ibin)
        
        directory = "fit_params_"+str(Ns)+"c_m"+mass+"_"+desc+"/"
        
        # clean-room Veff.h5 is WINDOWED [iblo, ibhi] (get_potential_jk only wrote the
        # double-well betas). Place each resample's minima_data at its ABSOLUTE ibeta so
        # ibetac stays an absolute index (matches the shooter's dibeta = ibetac_jk - ibetac0).
        # Outside the window -> NaN -> +inf in the objective, so the argmin never selects it.
        lis = np.full((1001, 4), np.nan)
        f = h5py.File(directory+"/Veff.h5", 'r')
        # print( f.keys() )
        for ibeta in ibetas:
            lis[ibeta] = f[str(ibeta)+'/minima_data'][()]
            # tmp = np.loadtxt( directory+"minima_data_"+str(ibeta)+".dat" )
        f.close()

        obj = (lis.T[2] - lis.T[3])**2
        obj = np.where(np.isnan(obj), np.inf, obj)

        ibeta_m_=ibeta_m-1
        ibeta_M_=ibeta_M-1

        ibetac=ibeta_m_
        while ibetac==ibeta_m_:
            ibeta_m_+=1
            ibeta_M_+=1
            ibetac = np.argmin( obj[ibeta_m_:ibeta_M_] ) + ibeta_m_
        if ibeta_m_==ibetac or abs(ibeta_M_-ibetac)<=4:
            print( jdrop, ibin )
            plt.plot( betas, lis.T[0])
            plt.plot( betas, lis.T[1] )
            plt.vlines( (betas[ibetac]), -0.1, 0.2, ls='dashed' )

        tmp_.append( ibetac )
        plt.plot( betas, lis.T[0])
        plt.plot( betas, lis.T[1] )
        plt.vlines( (betas[ibetac]), -0.1, 0.2, ls='dashed' )
    ibetac_jk_.append(tmp_)

plt.show()

In [ ]:
ibetac_jk = np.array( ibetac_jk_ )

In [ ]:
np.savetxt( "ibetac_jk_mass"+mass+"_"+str(Ns)+".dat", ibetac_jk )

In [ ]:
# v16: MEAN betac_ibetac from the mean Veff (argmin (VA-VB)^2 in the bracket). Writes the [betac, ibetac]
# file that hist_spline_jk reads for ibetac0 (its last line = ibetac). Run once per mass (same mass block).
lis_m = np.full((1001, 4), np.nan)
fm = h5py.File("fit_params_"+str(Ns)+"c_m"+mass+"/Veff.h5", 'r')
for ibeta in ibetas:
    lis_m[ibeta] = fm[str(ibeta)+'/minima_data'][()]
fm.close()
obj_m = (lis_m.T[2] - lis_m.T[3])**2
obj_m = np.where(np.isnan(obj_m), np.inf, obj_m)
ibetac_mean = int( np.argmin( obj_m[ibeta_m:ibeta_M] ) + ibeta_m )
print("v16 mean betac:", mass, ibetac_mean, betas[ibetac_mean])
np.savetxt( "betac_ibetac_mass"+mass+"_"+str(Ns)+".dat", [betas[ibetac_mean], ibetac_mean] )

In [ ]:
ibetac_jk.size

In [ ]:
pwd

In [ ]:
betas[ibetac]

In [ ]:
ibetac

In [ ]:
# ibeta_m = 775
# ibeta_M = 816

In [ ]:
# single-resample inspection (change jdrop/ibin to look at any resample)
jdrop = 3
ibin = 11

desc = "jk_"+str(jdrop)+"_40_"+str(ibin)

directory = "fit_params_"+str(Ns)+"c_m"+mass+"_"+desc+"/"

# clean-room: read the eps-shrunk Veff.h5, absolute-indexed (NaN outside the window)
lis = np.full((1001, 4), np.nan)
f = h5py.File(directory+"/Veff.h5", 'r')
for ibeta in ibetas:
    lis[ibeta] = f[str(ibeta)+'/minima_data'][()]
f.close()

obj = (lis.T[2] - lis.T[3])**2
obj = np.where(np.isnan(obj), np.inf, obj)

ibeta_m_=ibeta_m-1
ibeta_M_=ibeta_M-1

ibetac=ibeta_m_
while ibetac==ibeta_m_:
    ibeta_m_+=1
    ibeta_M_+=1
    ibetac = np.argmin( obj[ibeta_m_:ibeta_M_] ) + ibeta_m_
if ibeta_m_==ibetac or ibeta_M_-1==ibetac:
    print( jdrop, ibin )
    plt.plot( betas, lis.T[0])
    plt.plot( betas, lis.T[1] )
    plt.vlines( (betas[ibetac]), -0.1, 0.2, ls='dashed' )

# plt.vlines( (betas[783]), -0.1, 0.2, ls='dashed' )

plt.plot( betas, lis.T[0])
plt.plot( betas, lis.T[1] )
plt.vlines( (betas[ibetac]), -0.1, 0.2, ls='dashed' )
    # ibetac_jk_.append(tmp_)

In [ ]:
ibetac

In [ ]:
ibeta_m_

In [ ]:
plt.plot( betas, lis.T[2])
plt.plot( betas, lis.T[3] )
plt.vlines( (betas[ibeta_m_], betas[ibeta_M_]), -0.1, 0.1, ls='dashed' )
plt.vlines( (betas[ibetac]), -0.1, 0.1, ls='dashed', color='red' )

In [ ]:
ibeta_m, ibetac, ibeta_M